# Harmony integrated atlas analysis

In [ ]:
from pathlib import Path
from shared.repo import REPO_ROOT
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
atlaspath = Path().resolve().parents[1] / "output/atlas/v1/processed_1/atlas_harmony.h5ad"
adata = sc.read_h5ad(atlaspath, backed="r")
adata

In [ ]:
# sub = sc.pp.sample(adata, n=5000, copy=True)
subpath = Path().resolve().parents[1] / "output/atlas/v1/processed_1/atlas_harmony_sub.h5ad"
sub = sc.read_h5ad(subpath)
sub

In [ ]:
from shared.repo import REPO_ROOT
adata = sc.read_h5ad(REPO_ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX12366723.h5ad")
adata.obs.columns

In [ ]:
import anndata as ad
sub.write_h5ad(
    Path().resolve().parents[1] / "output/atlas/v1/processed_1/atlas_harmony_sub.h5ad",
    compression="gzip",
)

In [ ]:
sc.pl.embedding(
    sub,
    "X_umap_uncorrected",
    color=["study_accession", "cell_type", "leiden_uncorrected"],
    legend_loc=None,
    title=["Batch", "CxG Label", "Leiden clusters (uncorrected)"],
)

sc.pl.embedding(
    sub, "X_umap",
    color=["study_accession", "cell_type", "leiden_atlas"],
    legend_loc=None,
    title=["Batch", "CxG Label", "Leiden clusters (corrected)"],
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_uncorrected",
    layer="X_umap_uncorrected",
    legend_loc=None,
    alpha=0.5,
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_atlas",
    layer="X_umap",
    legend_loc=None,
    alpha=0.5,
)

## Compute metrics using `scib-metrics`
```
Adam Gayoso, Martin Kim, Ori Kronfeld, Justin Hong, & Yosef, N. (2026). YosefLab/scib-metrics: scib-metrics 0.5.8 (v0.5.8). Zenodo. https://doi.org/10.5281/zenodo.18504367
```

In [ ]:
sub

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

bm = Benchmarker(
    sub,
    batch_key="study_accession",
    label_key="cell_type",
    embedding_obsm_keys=["X_pca", "X_pca_harmony"],
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    pre_integrated_embedding_obsm_key="X_pca",
    n_jobs=6,
)

bm.benchmark()

bm.plot_results_table()